In [ ]:
import os
import pandas as pd
from umap import UMAP
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = r"E:/Data_for_Practice/JapMedia/"
embedding_csv_path = os.path.join(BASE_DIR, "all_topic_label_embeddings.csv")

df = pd.read_csv(embedding_csv_path, encoding="utf-8-sig")

df["newspaper"] = df["newspaper"].astype(str).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

emb_cols = [c for c in df.columns if c.startswith("emb_")]
if not emb_cols:
    raise RuntimeError("[ERROR] No embedding columns (emb_*) found in CSV.")

print("Years in embedding file:", df["year"].unique())

centroids = (
    df
    .groupby(["newspaper", "year"])
    [emb_cols]
    .mean()
    .reset_index()
)

print("\n=== Centroid groups ===")
print(centroids[["newspaper", "year"]])

X = centroids[emb_cols].values

umap_model = UMAP(
    n_neighbors=10,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
coords = umap_model.fit_transform(X)

centroids["x"] = coords[:, 0]
centroids["y"] = coords[:, 1]

# 🔹 연도별 색상 매핑
color_map_year = {
    2022: "#1f77b4",  # blue
    2023: "#ff7f0e",  # orange
    2024: "#2ca02c",  # green
}

plt.figure(figsize=(10, 8))

# 점만 표시 (도형 통일 → 원형)
for _, row in centroids.iterrows():
    yr = int(row["year"])
    plt.scatter(
        row["x"],
        row["y"],
        color=color_map_year.get(yr, "gray"),
        s=220,
        edgecolors="black",
        linewidths=0.9
    )
    plt.text(
        row["x"] + 0.03,
        row["y"] + 0.03,
        f"{row['newspaper']} {row['year']}",
        fontsize=9,
        weight="bold"
    )

# 연도 범례
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='white',
               label='2022', markerfacecolor=color_map_year[2022],
               markeredgecolor='black', markersize=10),
    plt.Line2D([0], [0], marker='o', color='white',
               label='2023', markerfacecolor=color_map_year[2023],
               markeredgecolor='black', markersize=10),
    plt.Line2D([0], [0], marker='o', color='white',
               label='2024', markerfacecolor=color_map_year[2024],
               markeredgecolor='black', markersize=10),
]
plt.legend(handles=legend_handles, title="Year")

plt.title("Centroid UMAP of Climate Topic Labels by Year (No Trajectory Lines)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.grid(alpha=0.15)
plt.tight_layout()

out_fig = os.path.join(BASE_DIR, "umap_centroid_by_year_no_lines.png")
plt.savefig(out_fig, dpi=300)
print(f"[FIG SAVED] {out_fig}")

plt.show()

In [ ]:
import os
import pandas as pd
from umap import UMAP
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3D projection

# 한글 폰트(Windows)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = r"E:/Data_for_Practice/JapMedia/"
embedding_csv_path = os.path.join(BASE_DIR, "all_topic_label_embeddings.csv")

df = pd.read_csv(embedding_csv_path, encoding="utf-8-sig")

df["newspaper"] = df["newspaper"].astype(str).str.strip()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

emb_cols = [c for c in df.columns if c.startswith("emb_")]
if not emb_cols:
    raise RuntimeError("[ERROR] No embedding columns found!")

print("Years in embedding file:", df["year"].unique())

# ====== 신문사 × 연도별 센트로이드 ======
centroids = (
    df
    .groupby(["newspaper", "year"])
    [emb_cols]
    .mean()
    .reset_index()
)

X = centroids[emb_cols].values

# ====== 3D UMAP ======
umap_model = UMAP(
    n_neighbors=10,
    min_dist=0.1,
    n_components=3,
    metric="cosine",
    random_state=42,
)
coords = umap_model.fit_transform(X)

centroids["x"] = coords[:, 0]
centroids["y"] = coords[:, 1]
centroids["z"] = coords[:, 2]

# ====== 색 (연도별) ======
color_map_year = {
    2022: "#1f77b4",  # blue
    2023: "#ff7f0e",  # orange
    2024: "#2ca02c",  # green
}

# ====== 3D Plot ======
fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")

for _, row in centroids.iterrows():
    yr = int(row["year"])
    ax.scatter(
        row["x"], row["y"], row["z"],
        color=color_map_year.get(yr, "gray"),
        s=160,
        edgecolors="black",
        linewidths=0.8
    )
    ax.text(
        row["x"], row["y"], row["z"],
        f"{row['newspaper']} {row['year']}",
        fontsize=8,
        weight="bold"
    )

# ====== 범례(연도) ======
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='white',
               label='2022', markerfacecolor=color_map_year[2022],
               markeredgecolor='black', markersize=10),
    plt.Line2D([0], [0], marker='o', color='white',
               label='2023', markerfacecolor=color_map_year[2023],
               markeredgecolor='black', markersize=10),
    plt.Line2D([0], [0], marker='o', color='white',
               label='2024', markerfacecolor=color_map_year[2024],
               markeredgecolor='black', markersize=10),
]
ax.legend(handles=legend_handles, title="Year", loc="upper left")

ax.set_title("3D UMAP of Climate Topic Label Centroids by Year")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.set_zlabel("UMAP-3")

# 회전 가능하게
ax.view_init(elev=10, azim=45)

plt.tight_layout()
out_fig = os.path.join(BASE_DIR, "umap_centroid_by_year_3D.png")
plt.savefig(out_fig, dpi=300)
print(f"[FIG SAVED] {out_fig}")

plt.show()
